# 60) Bartlett ve Levene Testleri
Bir önceki konuda (Varyansların Homojenliği) "iki grubun varyansı eşit mi?" sorusunu sormamız gerektiğini öğrendik. Bu iki test, bu soruyu **resmi olarak** cevaplayan iki farklı yöntemdir.

## Ortak Hipotezler (İkisi de Aynı Soruyu Sorar)
- **H0:** Grupların varyansları eşittir (homojendir)
- **H1:** Grupların varyansları eşit değildir

## Bartlett Testi
Daha **eski ve klasik** yöntemdir. Ama önemli bir zayıflığı var: **verinin normal dağıldığı varsayımına çok duyarlıdır** eğer veri normal değilse, Bartlett testi yanıltıcı sonuç verebilir (yanlışlıkla "varyanslar farklı" diyebilir, aslında öyle olmasa bile).

## Levene Testi
Daha **modern ve dayanıklı (robust)** bir alternatiftir. Normallikten sapmalara karşı çok daha az duyarlıdır bu yüzden **pratikte, hangisini kullanacağımızdan emin değilsek, Levene testini tercih edebiliriz.**

## Hangisini Ne Zaman Kullanmalı?
- Veri **normal dağılıyorsa** (Shapiro-Wilk ile önce kontrol edilmiş, Konu 55) → Bartlett de kullanılabilir, ama Levene yine de güvenlidir
- Veri **normal dağılmıyorsa veya emin değilsek** → **Levene** kullanırız (daha güvenli varsayılan seçim).

## Python'da Kullanımı
```python
from scipy import stats

# Levene Testi
stat, p = stats.levene(grup1, grup2)

# Bartlett Testi
stat, p = stats.bartlett(grup1, grup2)
```

## Karar Kuralı (Her Zamanki Gibi)
- p < α → H0 reddedilir → Varyanslar **eşit değil** → T testinde `equal_var=False` (Welch) kullan
- p ≥ α → H0 reddedilemedi → Varyanslar **eşit kabul edilebilir** → `equal_var=True` (Pooled) kullanılabilir

## Zincirin Tamamı (Büyük Resim)
$$\text{Shapiro-Wilk (Normallik)} \to \text{Levene/Bartlett (Varyans Homojenliği)} \to \text{equal\_var kararı} \to \text{T Testi}$$

Bu, gerçek bir analizde izlemen gereken **doğru sıradır**. Testi uygulamadan önce varsayımları kontrol etmek, sonucun güvenilirliğini artırır.

In [1]:
import numpy as np 
from scipy import stats 
kreatif_A = np.array([320, 285, 410, 295, 350, 275, 390, 310, 265, 335, 300, 355])
kreatif_B = np.array([245, 260, 230, 275, 250, 210, 265, 240, 255, 220, 235, 248])

# Test 1 - Kreatif A için:
# H0: Kreatif A grubu normal dağılıma uymaktadır.
# H1: Kreatif A grubu normal dağılıma uymamaktadır.

# Test 2 - Kreatif B için:
# H0: Kreatif B grubu normal dağılıma uymaktadır.
# H1: Kreatif B grubu normal dağılıma uymamaktadır.
kreatif_a_normallik = stats.shapiro(kreatif_A)[1]
kreatif_b_normallik = stats.shapiro(kreatif_B)[1]
print(f'Kreatif A grubu normallik testine ait P değeri: {kreatif_a_normallik}')
print(f'Kreatif B grubu normallik testine ait P değeri: {kreatif_b_normallik}')

# H0: Kreatif A ve Kreatif B'nin varyansları homojendir.
# H1: Kreatif A ve Kreatif B'nin varyansları homojen değildir.
levene = stats.levene(kreatif_A, kreatif_B)
bartlett = stats.bartlett(kreatif_A, kreatif_B)

print(f'Levene Testine ait P değeri: {levene.pvalue}')
print(f'Bartlett Testine ait P değeri: {bartlett.pvalue}')

Kreatif A grubu normallik testine ait P değeri: 0.6332660611032428
Kreatif B grubu normallik testine ait P değeri: 0.9997613703112531


ValueError: cannot convert float NaN to integer

### Önemli
SciPy kütüphanesinin en son sürümlerinden (1.18.0) kullandığımız için bartlett kısmında ufak bir sıkıntı yaşandı biz levene testi sonucunda elde edilen p değeri üzerinden (0.0196) dolayısıyla equal_var=False parametresini kullanacağız.

In [2]:
ttest = stats.ttest_ind(kreatif_A, kreatif_B, equal_var=False, alternative='two-sided')
alpha = 0.05
if (p_value:=ttest.pvalue) < alpha:
    print('H0 hipotezini reddediyoruz, Kreatif A ile Kreatif B"nin ortalamaları arasında anlamlı bir fark var.')
else:
    print('Elimizde H0 hipotezini reddecek kadar güçlü bir kanıt yok, bu durumda Kreatif A ile Kreatif B"nin ortalamaları arasında anlamlı bir fark olmadığı söylenebilir.')
print(f"P değeri: {round(p_value,4)}")
print(f"T istatistiği: {ttest.statistic}")
ci = ttest.confidence_interval(confidence_level=0.95)
print(f"Güven aralığı alt sınır: {ci.low}, Güven aralığı üst sınır: {ci.high}") # ttest objesi üzerinden güven aralığı fonksiyonu ile istediğimiz alfa değeri üzerinden güven aralığı hesaplayıp sonucu ci isimli güven aralığı nesnesine attık. Bu nesne üzerinden de low ve high niteliklerini kullanarak alt ve ust sınırı yazdırıyoruz.
print(f"Serbestlik Derecesi: {ttest.df}")
print(f'Kreatif A ortalama: {np.mean(kreatif_A)}')
print(f'Kreatif B ortalama: {np.mean(kreatif_B)}')
print(f'Kreatif A Standart Sapma: {np.std(kreatif_A)}')
print(f'Kreatif B Standart Sapma: {np.std(kreatif_B)}')
print(f'Kreatif A ortalama - Kreatif B ortalama = {np.mean(kreatif_A) - np.mean(kreatif_B)}')

H0 hipotezini reddediyoruz, Kreatif A ile Kreatif B"nin ortalamaları arasında anlamlı bir fark var.
P değeri: 0.0001
T istatistiği: 5.63954152422423
Güven aralığı alt sınır: 49.546853010183106, Güven aralığı üst sınır: 109.95314698981696
Serbestlik Derecesi: 14.65538147222946
Kreatif A ortalama: 324.1666666666667
Kreatif B ortalama: 244.41666666666666
Kreatif A Standart Sapma: 43.34134541315281
Kreatif B Standart Sapma: 17.923254602765525
Kreatif A ortalama - Kreatif B ortalama = 79.75000000000003


### Sonuç
Bir test istatistiğini uygulamadan önce o testin varsayımlarını, kullanacağımız parametrelerin anlamlarını iyice araştırıp işlememiz daha doğru sonuçlar elde etmemiz açısından önemli. Biz de bu örnekte öncelikle varyansların homojenliğini default değerinde bırakmamıza rağmen ardından bu sayfada bu varsayımı test ederek daha doğru sonuçlar elde ettik.